# S3 — 分布偵探：Distribution & Outlier Analysis

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（類別型）、`datasets/house-prices/train.csv`（數值型）  
> **學完能幹嘛**：看穿每個欄位的分布特性，找出離群值，判斷哪些特徵可能對預測有用

---

## 為什麼要深入看分布？

S1 的 First Look 告訴你「有什麼」，S2 處理了缺值，現在要問更深的問題：

- 這個欄位的分布「正常」嗎？
- 不同類別對 target 的影響一樣嗎？
- 有沒有離群值在搞破壞？

**一句話口訣：每個欄位都有自己的故事 — 偏態告訴你資料的脾氣，離群值告訴你資料的秘密。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/titanic/train.csv')
hp = pd.read_csv('datasets/house-prices/train.csv')
print(f'Titanic: {df.shape} | House Prices: {hp.shape}')

---
## 核心觀念 1／3 — 數值型分布三板斧：Histogram + KDE + Boxplot

每種圖看不同東西：

| 圖表 | 看什麼 | 適合 |
|:-----|:-------|:-----|
| Histogram | 整體分布形狀 | 偏態、多峰 |
| KDE | 平滑的密度曲線，可疊加比較 | 比較不同群組 |
| Boxplot | 中位數、四分位、離群值 | 快速偵測離群值 |

In [ ]:
# 技巧 1：一行看完所有數值欄位的 histogram
hp.select_dtypes(include='number').hist(figsize=(16, 12), bins=30, edgecolor='white')
plt.suptitle('House Prices — 所有數值欄位分布', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print('→ 一眼看出：SalePrice、LotArea、GrLivArea 明顯右偏')

In [ ]:
# 技巧 2：KDE 疊圖 — 比較不同群組的分布差異
def plot_kde_by_target(df, feature, target, title=None, figsize=(8, 5)):
    """繪製 KDE 圖，按 target 分群疊加"""
    fig, ax = plt.subplots(figsize=figsize)
    for val in sorted(df[target].dropna().unique()):
        subset = df[df[target] == val][feature].dropna()
        sns.kdeplot(subset, fill=True, ax=ax, label=f'{target}={val}')
    ax.set_title(title or f'{feature} by {target}', fontsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_kde_by_target(df, 'Age', 'Survived', 'Age 分布：存活 vs 死亡')
print('→ 幼兒（< 10 歲）存活比例明顯較高 — 婦孺優先！')

In [ ]:
plot_kde_by_target(df, 'Fare', 'Survived', 'Fare 分布：存活 vs 死亡')
print('→ 高票價乘客存活率更高 — 1 等艙有更多救生艇機會')

In [ ]:
# 技巧 3：偏態量化
print('House Prices 數值欄位偏態 (skewness) — 前 10 名：')
skew_vals = hp.select_dtypes(include='number').skew().sort_values(ascending=False)
print(skew_vals.head(10))
print('\n→ skew > 1 代表嚴重右偏，考慮 log 轉換')

In [ ]:
# 技巧 4：log 轉換修正偏態
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].set_title('SalePrice 原始分布', fontsize=12)
sns.histplot(hp['SalePrice'], bins=30, kde=True, ax=axes[0], color='steelblue')

axes[1].set_title('SalePrice log 轉換後', fontsize=12)
sns.histplot(np.log1p(hp['SalePrice']), bins=30, kde=True, ax=axes[1], color='coral')

plt.tight_layout()
plt.show()
print(f'原始 skew: {hp["SalePrice"].skew():.2f} → log 後: {np.log1p(hp["SalePrice"]).skew():.2f}')
print('→ log 轉換後接近常態分布，模型表現通常更好')

**口訣**：histogram 看全貌、KDE 疊圖比群組、skew 量化偏態程度 — 右偏就 `np.log1p`。

> ### 💡 知識補給站 — 為什麼用 log1p 而不是 log？
> 
> `np.log(0)` 會得到 `-inf`（負無窮大），而 `np.log1p(0) = np.log(1+0) = 0`。
> 
> 資料中可能有 0 值（例如：沒有泳池的房子 PoolArea = 0），所以用 `log1p` 更安全。
> 反轉換用 `np.expm1`：`np.expm1(np.log1p(x)) == x`。
> 
> *延伸關鍵字：log transformation, log1p, expm1, Box-Cox transformation*

---
## 核心觀念 2／3 — 類別型分析：交叉比較

類別型欄位的核心問題：**這個類別對 target 有區分度嗎？**

In [ ]:
# 技巧 1：堆疊長條圖 — 各類別的 target 分布
def bar_chart_stacked(df, feature, target='Survived', stacked=True):
    """繪製堆疊/群組長條圖，比較各類別的 target 分布"""
    survived = df[df[target] == 1][feature].value_counts()
    dead = df[df[target] == 0][feature].value_counts()
    compare = pd.DataFrame({'Survived': survived, 'Dead': dead})
    ax = compare.plot(kind='bar', stacked=stacked, figsize=(8, 5),
                      color=['#66b2ff', '#ff9999'], edgecolor='white')
    ax.set_title(f'{feature} vs {target}', fontsize=14)
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

bar_chart_stacked(df, 'Sex')
print('→ 女性存活率遠高於男性 — Sex 是最強特徵之一')

In [ ]:
bar_chart_stacked(df, 'Pclass')
print('→ 1 等艙存活率最高，3 等艙最低 — 貧富差距在災難中放大')

In [ ]:
# 技巧 2：多變量分組比較
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='Sex', y='Survived', hue='Pclass', data=df, ax=ax)
ax.set_title('存活率 by Sex × Pclass', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

print('→ 即使同為女性，3 等艙的存活率也明顯低於 1、2 等艙')

In [ ]:
# 技巧 3：crosstab 快速看比例
print('Sex × Survived 交叉比例（按 Sex 正規化）：')
print(pd.crosstab(df['Sex'], df['Survived'], normalize='index').round(3))
print('\n→ 女性 74.2% 存活，男性只有 18.9%')

In [ ]:
# 技巧 4：Swarm Plot — 看雙變量的細節分布
def plot_swarm(df, x, y, hue='Survived', title=None):
    """繪製 swarm plot 觀察雙變量分布"""
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.swarmplot(data=df, x=x, y=y, hue=hue, ax=ax, size=3)
    ax.set_title(title or f'{y} by {x}', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_swarm(df, 'Sex', 'Age', title='Age × Sex × Survived')
print('→ 男性中，年幼者（< 15 歲）存活率較高；女性普遍存活率高')

**口訣**：類別看 stacked bar，多變量用 `barplot(hue=)`，數字看 `crosstab(normalize)` — 找出對 target 有區分度的特徵。

---
## 核心觀念 3／3 — 離群值偵測與處理

離群值不一定要刪 — 有時它是 **真實的極端情況**，有時是 **資料錯誤**。

| 方法 | 公式 | 適用 |
|:-----|:-----|:-----|
| IQR | Q1 - 1.5×IQR ~ Q3 + 1.5×IQR | 通用，不受偏態影響 |
| Z-score | \|z\| > 3 | 近似常態分布 |
| 視覺化 | scatter plot | 永遠先看圖 |

In [ ]:
# 經典案例：House Prices 的 GrLivArea 離群值
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(hp['GrLivArea'], hp['SalePrice'], alpha=0.5, s=20)
ax.set_xlabel('GrLivArea (地上面積 sqft)', fontsize=12)
ax.set_ylabel('SalePrice', fontsize=12)
ax.set_title('GrLivArea vs SalePrice — 找出離群值', fontsize=14)

# 標記可疑離群值
outliers = hp[(hp['GrLivArea'] > 4000)]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'],
           color='red', s=100, zorder=5, label=f'離群值 ({len(outliers)} 筆)')
ax.legend()
plt.tight_layout()
plt.show()

print('→ 右下角 2 個點：面積超大但價格很低 — 這是 Kaggle 公認的離群值')
print('→ 刪掉這 2 點通常可以提升模型分數')

In [ ]:
# IQR 方法
def detect_outliers_iqr(series):
    """用 IQR 方法偵測離群值"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = (series < lower) | (series > upper)
    return outliers, lower, upper

outlier_mask, lower, upper = detect_outliers_iqr(hp['SalePrice'])
print(f'SalePrice IQR 離群值偵測：')
print(f'  下界: {lower:,.0f} | 上界: {upper:,.0f}')
print(f'  離群值數量: {outlier_mask.sum()} / {len(hp)} ({outlier_mask.mean():.1%})')

In [ ]:
# np.clip — 「截斷」離群值而非刪除
sale_clipped = np.clip(hp['SalePrice'], lower, upper)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=hp['SalePrice'], ax=axes[0], color='steelblue')
axes[0].set_title('原始 SalePrice', fontsize=12)

sns.boxplot(x=sale_clipped, ax=axes[1], color='coral')
axes[1].set_title('clip 後 SalePrice', fontsize=12)

plt.tight_layout()
plt.show()
print('→ clip 保留了所有資料筆數，只是把極端值「壓回」邊界')

**口訣**：先用 scatter plot 看全貌，IQR 量化找離群值，`np.clip` 截斷比刪除更安全。

> ### 💡 知識補給站 — 離群值處理的領域知識
> 
> 統計方法告訴你「什麼是異常」，但只有領域知識能告訴你「該不該刪」：
> 
> - $500K 的房子 → 統計上異常，但在好地段很正常（**保留**）
> - 4500 sqft 的房子卻賣 $12K → 很可能是資料錯誤或特殊交易（**刪除**）
> - 年齡 150 歲 → 明顯資料錯誤（**修正或刪除**）
> 
> 在 Kaggle 比賽中，處理離群值通常需要看 Discussion 區的分析和 domain knowledge。
> 
> *延伸關鍵字：domain knowledge, outlier treatment, robust statistics, Winsorization*

---
## 實務 Case — 分箱技巧（連續變數 → 類別化）

有時候把連續變數「切段」反而更有分析價值。

In [ ]:
# qcut：等頻分箱
categories = ['cheap', 'standard', 'expensive', 'luxury']
df['Fare_bin'] = pd.qcut(df['Fare'], q=4, labels=categories)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='Fare_bin', y='Survived', data=df, ax=ax, order=categories)
ax.set_title('存活率 by 票價分組 (qcut)', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

print('→ 票價越高存活率越高，從 cheap 的 ~20% 到 luxury 的 ~58%')

---
## 課堂練習（40 min）

### 🟢 送分題

用 House Prices 資料集，**一行程式碼**畫出所有數值欄位的 histogram。  
（提示：`df.select_dtypes(include='number').hist(...)`）

In [ ]:
# TODO: 你的程式碼

### 🟡 核心題

1. 找出 House Prices 中 **偏態 (skew) 前 3 名** 的數值欄位
2. 對這 3 個欄位分別做 `np.log1p` 轉換
3. 畫出轉換前後的 KDE 圖（3 個欄位 × 2 = 6 張子圖）

In [ ]:
# TODO: 你的程式碼

### 🔴 挑戰題

用 Titanic 資料集：
1. 建立 `Age_bin`，用 `pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 80])` 分成 5 組
2. 畫出各年齡組的存活率 barplot
3. 加上 `Sex` 作為 hue，畫出 Age_bin × Sex × Survived 的交互圖
4. 寫下你從這張圖觀察到的 2-3 個發現

In [ ]:
# TODO: 你的程式碼

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 全部 histogram | `df.select_dtypes(include='number').hist()` |
| KDE 疊圖 | `sns.kdeplot(subset, fill=True, label=...)` |
| Boxplot | `sns.boxplot(x=df['col'])` |
| 偏態值 | `df.skew().sort_values(ascending=False)` |
| Log 轉換 | `np.log1p(df['col'])` |
| 堆疊長條圖 | `compare.plot(kind='bar', stacked=True)` |
| 群組比較 | `sns.barplot(x=feat, y=target, hue=group)` |
| 交叉表 | `pd.crosstab(df['a'], df['b'], normalize='index')` |
| Swarm plot | `sns.swarmplot(data=df, x=x, y=y, hue=target)` |
| IQR 離群值 | `Q1 - 1.5*IQR ~ Q3 + 1.5*IQR` |
| 截斷離群值 | `np.clip(series, lower, upper)` |
| 等頻分箱 | `pd.qcut(df['col'], q=4, labels=[...])` |

**下節預告 S4**：現在你知道每個欄位的「脾氣」了，接下來要把原始欄位變成更有用的特徵 → **Feature Engineering Thinking**